# 04 - E91 con Eve e sorgente classica


In questo notebook confrontiamo tre scenari legati al protocollo E91.

Il caso ideale usa coppie entangled condivise tra Alice e Bob. L'attacco intercept-resend di Eve rompe o degrada le correlazioni quantistiche. Una sorgente classicamente correlata pu? generare correlazioni nei bit, ma non deve produrre una violazione CHSH robusta.

Il confronto usa due indicatori: il QBER sulla chiave sifted e il parametro CHSH `S`.

## Setup

Prepariamo gli import, aggiungiamo `src` al path e creiamo le cartelle in cui salvare tabelle e figure.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

current_path = Path.cwd()

if (current_path / "src" / "e91.py").exists():
    project_root = current_path
else:
    project_root = current_path.parent

src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from e91 import (
    run_e91_protocol,
    run_e91_protocol_with_eve,
    run_e91_protocol_with_classical_source,
    run_chsh_experiment,
    run_chsh_experiment_with_eve,
    run_chsh_experiment_with_classical_source,
)
from qkd_core import sift_keys, compute_qber
from metrics import (
    QBER_THRESHOLD_BB84,
    CHSH_CLASSICAL_LIMIT,
    CHSH_TSIRELSON_BOUND,
    compute_sifted_key_length,
    compute_sifted_key_rate,
    decide_qber_acceptance,
)

tables_dir = project_root / "results" / "tables"
figures_dir = project_root / "results" / "figures"

tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print("Setup completato.")

## Parametri

Impostiamo il numero di round, gli shot CHSH, il seed e le probabilit? di intercettazione di Eve.

In [ ]:
N_ROUNDS = 1000
SHOTS_CHSH = 2000
SEED = 42
INTERCEPT_PROBABILITIES = [0.0, 0.25, 0.5, 0.75, 1.0]
QBER_THRESHOLD = QBER_THRESHOLD_BB84

print("Numero di round:", N_ROUNDS)
print("Shot CHSH:", SHOTS_CHSH)
print("Seed:", SEED)
print("Probabilit? di intercettazione:", INTERCEPT_PROBABILITIES)
print("Soglia QBER:", QBER_THRESHOLD)

## Scenario E91 ideale

Eseguiamo E91 ideale, calcoliamo la chiave sifted, il QBER e il test CHSH.

In [ ]:
results_ideal = run_e91_protocol(N_ROUNDS, seed=SEED)

alice_key_ideal, bob_key_ideal = sift_keys(results_ideal)
qber_ideal = compute_qber(alice_key_ideal, bob_key_ideal)
sifted_key_length_ideal = compute_sifted_key_length(alice_key_ideal)
sifted_key_rate_ideal = compute_sifted_key_rate(alice_key_ideal, N_ROUNDS)
chsh_ideal = run_chsh_experiment(shots=SHOTS_CHSH, seed=SEED)

print("E91 ideale")
print("  lunghezza chiave sifted:", sifted_key_length_ideal)
print("  sifted key rate:", sifted_key_rate_ideal)
print("  QBER:", qber_ideal)
print("  |S|:", chsh_ideal["abs_S"])

## Scenario E91 con Eve sempre attiva

Eseguiamo E91 con Eve intercept-resend attiva in ogni round e ripetiamo le stesse misure.

In [ ]:
results_eve = run_e91_protocol_with_eve(
    N_ROUNDS,
    intercept_probability=1.0,
    seed=SEED,
)

alice_key_eve, bob_key_eve = sift_keys(results_eve)
qber_eve = compute_qber(alice_key_eve, bob_key_eve)
sifted_key_length_eve = compute_sifted_key_length(alice_key_eve)
sifted_key_rate_eve = compute_sifted_key_rate(alice_key_eve, N_ROUNDS)
chsh_eve = run_chsh_experiment_with_eve(
    intercept_probability=1.0,
    shots=SHOTS_CHSH,
    seed=SEED,
)

print("E91 con Eve")
print("  lunghezza chiave sifted:", sifted_key_length_eve)
print("  sifted key rate:", sifted_key_rate_eve)
print("  QBER:", qber_eve)
print("  |S|:", chsh_eve["abs_S"])

## Scenario sorgente classica

Eseguiamo E91 con una sorgente classicamente correlata, utile come confronto con il caso entangled.

In [ ]:
results_classical = run_e91_protocol_with_classical_source(N_ROUNDS, seed=SEED)

alice_key_classical, bob_key_classical = sift_keys(results_classical)
qber_classical = compute_qber(alice_key_classical, bob_key_classical)
sifted_key_length_classical = compute_sifted_key_length(alice_key_classical)
sifted_key_rate_classical = compute_sifted_key_rate(alice_key_classical, N_ROUNDS)
chsh_classical = run_chsh_experiment_with_classical_source(
    shots=SHOTS_CHSH,
    seed=SEED,
)

print("E91 con sorgente classica")
print("  lunghezza chiave sifted:", sifted_key_length_classical)
print("  sifted key rate:", sifted_key_rate_classical)
print("  QBER:", qber_classical)
print("  |S|:", chsh_classical["abs_S"])

## Tabelle dei round

Salviamo i round dei tre scenari. Per il caso ideale usiamo un nome distinto da quello del notebook 03.

In [ ]:
df_e91_ideal = pd.DataFrame(results_ideal)
df_e91_eve = pd.DataFrame(results_eve)
df_e91_classical = pd.DataFrame(results_classical)

ideal_rounds_path = tables_dir / "e91_ideal_rounds_notebook04.csv"
eve_rounds_path = tables_dir / "e91_eve_rounds.csv"
classical_rounds_path = tables_dir / "e91_classical_source_rounds.csv"

df_e91_ideal.to_csv(ideal_rounds_path, index=False)
df_e91_eve.to_csv(eve_rounds_path, index=False)
df_e91_classical.to_csv(classical_rounds_path, index=False)

print(f"Round E91 ideale salvati in: {ideal_rounds_path}")
print(f"Round E91 con Eve salvati in: {eve_rounds_path}")
print(f"Round sorgente classica salvati in: {classical_rounds_path}")

df_e91_ideal.head(10)

In [ ]:
df_e91_eve.head(10)

In [ ]:
df_e91_classical.head(10)

## Tabella confronto QBER

Confrontiamo QBER, lunghezza della chiave sifted e accettazione rispetto alla soglia didattica.

In [ ]:
qber_comparison = [
    {
        "scenario": "E91 ideal",
        "n_rounds": N_ROUNDS,
        "intercept_probability": None,
        "sifted_key_length": sifted_key_length_ideal,
        "sifted_key_rate": sifted_key_rate_ideal,
        "qber": qber_ideal,
        "qber_threshold": QBER_THRESHOLD,
        "accepted": decide_qber_acceptance(qber_ideal, QBER_THRESHOLD),
        "seed": SEED,
    },
    {
        "scenario": "E91 Eve intercept-resend",
        "n_rounds": N_ROUNDS,
        "intercept_probability": 1.0,
        "sifted_key_length": sifted_key_length_eve,
        "sifted_key_rate": sifted_key_rate_eve,
        "qber": qber_eve,
        "qber_threshold": QBER_THRESHOLD,
        "accepted": decide_qber_acceptance(qber_eve, QBER_THRESHOLD),
        "seed": SEED,
    },
    {
        "scenario": "E91 classical source",
        "n_rounds": N_ROUNDS,
        "intercept_probability": None,
        "sifted_key_length": sifted_key_length_classical,
        "sifted_key_rate": sifted_key_rate_classical,
        "qber": qber_classical,
        "qber_threshold": QBER_THRESHOLD,
        "accepted": decide_qber_acceptance(qber_classical, QBER_THRESHOLD),
        "seed": SEED,
    },
]

qber_comparison_df = pd.DataFrame(qber_comparison)
qber_comparison_path = tables_dir / "e91_attack_classical_qber_comparison.csv"
qber_comparison_df.to_csv(qber_comparison_path, index=False)

print(f"Confronto QBER salvato in: {qber_comparison_path}")
qber_comparison_df

## Tabella confronto CHSH

Confrontiamo il parametro CHSH nei tre scenari.

In [ ]:
chsh_comparison = [
    {
        "scenario": "E91 ideal",
        "shots": SHOTS_CHSH,
        "S": chsh_ideal["S"],
        "abs_S": chsh_ideal["abs_S"],
        "violates_chsh": chsh_ideal["violates_chsh"],
        "chsh_gap": chsh_ideal["chsh_gap"],
        "chsh_strength": chsh_ideal["chsh_strength"],
        "intercept_probability": None,
        "source_type": "entangled",
        "seed": SEED,
    },
    {
        "scenario": "E91 Eve intercept-resend",
        "shots": SHOTS_CHSH,
        "S": chsh_eve["S"],
        "abs_S": chsh_eve["abs_S"],
        "violates_chsh": chsh_eve["violates_chsh"],
        "chsh_gap": chsh_eve["chsh_gap"],
        "chsh_strength": chsh_eve["chsh_strength"],
        "intercept_probability": 1.0,
        "source_type": "entangled_with_eve",
        "seed": SEED,
    },
    {
        "scenario": "E91 classical source",
        "shots": SHOTS_CHSH,
        "S": chsh_classical["S"],
        "abs_S": chsh_classical["abs_S"],
        "violates_chsh": chsh_classical["violates_chsh"],
        "chsh_gap": chsh_classical["chsh_gap"],
        "chsh_strength": chsh_classical["chsh_strength"],
        "intercept_probability": None,
        "source_type": "classical_correlated",
        "seed": SEED,
    },
]

chsh_comparison_df = pd.DataFrame(chsh_comparison)
chsh_comparison_path = tables_dir / "e91_attack_classical_chsh_comparison.csv"
chsh_comparison_df.to_csv(chsh_comparison_path, index=False)

print(f"Confronto CHSH salvato in: {chsh_comparison_path}")
chsh_comparison_df

## Sweep su Eve

Studiamo come cambiano QBER e CHSH al variare della probabilit? di intercettazione di Eve.

In [ ]:
sweep_rows = []

for i in range(len(INTERCEPT_PROBABILITIES)):
    probability = INTERCEPT_PROBABILITIES[i]

    results = run_e91_protocol_with_eve(
        N_ROUNDS,
        intercept_probability=probability,
        seed=SEED,
    )
    sweep_alice_key, sweep_bob_key = sift_keys(results)
    sweep_qber = compute_qber(sweep_alice_key, sweep_bob_key)
    sweep_sifted_key_length = compute_sifted_key_length(sweep_alice_key)
    sweep_sifted_key_rate = compute_sifted_key_rate(sweep_alice_key, N_ROUNDS)
    sweep_chsh = run_chsh_experiment_with_eve(
        intercept_probability=probability,
        shots=SHOTS_CHSH,
        seed=SEED,
    )

    row = {
        "intercept_probability": probability,
        "qber": sweep_qber,
        "sifted_key_length": sweep_sifted_key_length,
        "sifted_key_rate": sweep_sifted_key_rate,
        "abs_S": sweep_chsh["abs_S"],
        "violates_chsh": sweep_chsh["violates_chsh"],
        "chsh_gap": sweep_chsh["chsh_gap"],
        "chsh_strength": sweep_chsh["chsh_strength"],
        "seed": SEED,
    }
    sweep_rows.append(row)

    print("Probabilit?:", probability)
    print("  QBER:", sweep_qber)
    print("  |S|:", sweep_chsh["abs_S"])

sweep_df = pd.DataFrame(sweep_rows)
sweep_path = tables_dir / "e91_eve_qber_chsh_vs_interception.csv"
sweep_df.to_csv(sweep_path, index=False)

print(f"Sweep Eve salvato in: {sweep_path}")
sweep_df

## Grafico QBER confronto scenari

Visualizziamo il QBER dei tre scenari con la soglia didattica di riferimento.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

scenarios = ["E91 ideal", "E91 Eve intercept-resend", "E91 classical source"]
qber_values = qber_comparison_df["qber"]
colors = ["tab:blue", "tab:orange", "tab:green"]

bars = ax.bar(
    scenarios,
    qber_values,
    color=colors,
)

ax.axhline(
    QBER_THRESHOLD,
    linestyle="--",
    color="tab:red",
    label="Soglia QBER",
)

for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.3f}",
        ha="center",
        va="bottom",
    )

ax.set_title("E91: confronto QBER tra scenari")
ax.set_ylabel("QBER")
ax.set_ylim(0, max(0.3, qber_comparison_df["qber"].max() + 0.05))
ax.grid(axis="y", alpha=0.3)
ax.legend()

plt.xticks(rotation=20, ha="right")
fig.tight_layout()

qber_figure_path = figures_dir / "e91_attack_classical_qber_comparison.png"
fig.savefig(qber_figure_path, dpi=300)
plt.show()

print(f"Grafico QBER scenari salvato in: {qber_figure_path}")


## Grafico CHSH confronto scenari

Usiamo un dot plot: i punti sono risultati simulati, mentre le linee sono soglie teoriche.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

scenarios = ["E91 ideal", "E91 Eve intercept-resend", "E91 classical source"]
abs_s_values = chsh_comparison_df["abs_S"]
colors = ["tab:blue", "tab:orange", "tab:green"]

bars = ax.bar(
    scenarios,
    abs_s_values,
    color=colors,
)

ax.axhline(
    CHSH_CLASSICAL_LIMIT,
    linestyle="--",
    color="tab:red",
    label="Limite classico |S| = 2",
)
ax.axhline(
    CHSH_TSIRELSON_BOUND,
    linestyle="--",
    color="tab:purple",
    label="Limite quantistico |S| = 2\u221a2",
)

for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.3f}",
        ha="center",
        va="bottom",
    )

ax.set_title("E91: confronto CHSH tra scenari")
ax.set_ylabel("Valore di |S|")
ax.set_ylim(0, 3.1)
ax.grid(axis="y", alpha=0.3)
ax.legend()

plt.xticks(rotation=20, ha="right")
fig.tight_layout()

chsh_figure_path = figures_dir / "e91_attack_classical_chsh_comparison.png"
fig.savefig(chsh_figure_path, dpi=300)
plt.show()

print(f"Grafico CHSH scenari salvato in: {chsh_figure_path}")


## Grafici sweep Eve

Rappresentiamo QBER e |S| al variare della probabilit? di intercettazione.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(
    sweep_df["intercept_probability"],
    sweep_df["qber"],
    marker="o",
    label="QBER simulato",
)
ax.axhline(
    QBER_THRESHOLD,
    linestyle="--",
    color="tab:red",
    label="Soglia QBER",
)
ax.set_title("E91 con Eve: QBER vs probabilit? di intercettazione")
ax.set_xlabel("Probabilit? di intercettazione")
ax.set_ylabel("QBER")
ax.set_ylim(0, max(0.3, sweep_df["qber"].max() + 0.05))
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

sweep_qber_figure_path = figures_dir / "e91_eve_qber_vs_interception.png"
fig.savefig(sweep_qber_figure_path, dpi=300)
plt.show()

print(f"Grafico QBER sweep salvato in: {sweep_qber_figure_path}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(
    sweep_df["intercept_probability"],
    sweep_df["abs_S"],
    marker="o",
    label="|S| simulato",
)
ax.axhline(
    CHSH_CLASSICAL_LIMIT,
    linestyle="--",
    color="tab:red",
    label="Limite classico |S| = 2",
)
ax.axhline(
    CHSH_TSIRELSON_BOUND,
    linestyle="--",
    color="tab:green",
    label="Limite quantistico |S| = 2√2",
)
ax.set_title("E91 con Eve: CHSH vs probabilit? di intercettazione")
ax.set_xlabel("Probabilit? di intercettazione")
ax.set_ylabel("|S|")
ax.set_ylim(0, 3.1)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

sweep_chsh_figure_path = figures_dir / "e91_eve_chsh_vs_interception.png"
fig.savefig(sweep_chsh_figure_path, dpi=300)
plt.show()

print(f"Grafico CHSH sweep salvato in: {sweep_chsh_figure_path}")

## Commento finale

E91 ideale mostra QBER nullo e violazione CHSH. L'attacco intercept-resend di Eve aumenta il QBER e riduce la violazione CHSH.

La sorgente classica pu? produrre correlazione nei bit, ma non una violazione CHSH robusta. CHSH ? quindi il controllo specificamente quantistico che distingue l'entanglement reale da una correlazione classica.